# Data ingestion

In [2]:
import langchain_community.document_loaders as loaders
import os

In [7]:
class MultiFileLoader:
    """
    A more robust loader that can handle a directory of files with different extensions.
    """
    # A mapping from file extensions to their corresponding LangChain loader class.
    # This makes it easy to add support for new file types.
    LOADER_MAPPING = {
        ".pdf": loaders.PyPDFLoader,
        ".docx": loaders.Docx2txtLoader,
        ".txt": loaders.TextLoader, 
        ".csv": loaders.CSVLoader,
        ".json": loaders.JSONLoader,
    }

    def __init__(self, directory_path):
        """
        Initializes the loader with the path to the directory.
        """
        if not os.path.isdir(directory_path):
            raise ValueError(f"The path '{directory_path}' is not a valid directory.")
        self.directory_path = directory_path

    def load_documents(self):
        """
        Loads all supported documents from the specified directory.
        It iterates through the files, finds the correct loader based on the
        file extension, and loads the content.
        """
        documents = []
        print(f"Loading documents from '{self.directory_path}'...")
        for filename in os.listdir(self.directory_path):
            filepath = os.path.join(self.directory_path, filename)
            
            if os.path.isfile(filepath):
                _, extension = os.path.splitext(filename)
                loader_class = self.LOADER_MAPPING.get(extension.lower())

                if loader_class:
                    try:
                        print(f"-> Loading '{filename}' with {loader_class.__name__}")
                        loader = loader_class(filepath)
                        documents.extend(loader.load())
                    except Exception as e:
                        print(f"  [!] Error loading '{filename}': {e}")
                else:
                    print(f"-> Skipping unsupported file type: '{filename}'")
        
        print(f"Finished loading. Total documents loaded: {len(documents)}")
        return documents

In [8]:
# Now load the documents in the data directory 
documents = MultiFileLoader("../data").load_documents()

Loading documents from '../data'...
-> Loading '1706.03762v7.pdf' with PyPDFLoader
-> Loading '1810.04805v2.pdf' with PyPDFLoader
-> Loading '2005.14165v4.pdf' with PyPDFLoader
-> Loading '2106.09685v2.pdf' with PyPDFLoader
-> Loading '2203.02155v1.pdf' with PyPDFLoader
-> Loading '2302.13971v1.pdf' with PyPDFLoader
Finished loading. Total documents loaded: 227


In [9]:
# Let's inspect the first document to see its structure
if documents:
    print("--- First Document --- ")
    print(documents[0])
    print("\n--- Metadata Only --- ")
    print(documents[0].metadata)

--- First Document --- 
page_content='Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with rec

In [10]:
# Creating chunks of the data to be used for embeddings and vector store
class Chunker:
    """
    A class to chunk the loaded documents
    """
    def __init__(self, chunk_size=1000, chunk_overlap=0):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def chunk_documents(self, documents):
        from langchain_text_splitters import RecursiveCharacterTextSplitter

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )
        return splitter.split_documents(documents)

In [18]:
chunked_documents = Chunker().chunk_documents(documents)
chunked_documents[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data\\1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Bra

### Embedings and Vector DB

In [5]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import uuid
from typing import List,Any
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
class EmbeddingManager:
    """ 
    A class to manage the embedding of the text
    """
    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.__load_model()

    def __load_model(self):
        """
        Load the embedding model
        """
        try:
            # Setting device="cuda" to utilize the RTX 4050 for significantly faster encoding
            self.model = SentenceTransformer(self.model_name, device="cuda")
            print(f"Successfully loaded the model {self.model_name} on {self.model.device}")
            print(f"Embedding dimension: {self.model.get_embedding_dimension()}")      

        except Exception as e:
            print(f"Error occurred while loading the model: {e}")

    def embd_documents(self, documents: List[Any]) -> np.ndarray:
        """
        Embed the documents and return the embeddings vectors
        """
        # Extract string content from the LangChain Document objects before encoding
        texts = [doc.page_content for doc in documents]
        
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding of shape {embeddings.shape}")
        return embeddings
    
# Initialize the embedding manager
embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2591.27it/s]


Successfully loaded the model all-MiniLM-L6-v2 on cuda:0
Embedding dimension: 384


In [20]:
class VectorStoreManager:
    """ 
    Class to manage the storage of the converted vectors 
    """
    def __init__(self, collection_name: str="pdf_files", directory: str="../data/vector_store"):
        self.collection_name = collection_name
        self.directory = directory
        self.client = None
        self.collection = None
        self.__initialize_store()

    def __initialize_store(self):
        """
        Initialize the collection
        """
        self.client = chromadb.PersistentClient(path=self.directory)
        try:
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={ # Configure ChromaDB to use cosine similarity
                    "hnsw:space": "cosine",
                    "description": "pdf documents embedded for the RAG"
                }
            )

        except Exception as e:
            print(f"Error initializing store: {e}")
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of embeddings must match the number of documents")
            
        print(f"Adding {len(documents)} documents to vector space...")
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Create a uuid for better uniqueness
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Add metadata 
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["context_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Add text
            documents_texts.append(doc.page_content)

            # Add embedding
            embeddings_list.append(embedding.tolist())

        try:
            # Batch inserts to respect ChromaDB constraints on massive datasets
            batch_size = 5000
            for i in range(0, len(ids), batch_size):
                self.collection.add(
                        ids=ids[i:i+batch_size],
                        embeddings=embeddings_list[i:i+batch_size],
                        metadatas=metadatas[i:i+batch_size],
                        documents=documents_texts[i:i+batch_size]
                    )
            print("Successfully added the required fields in client")
                
        except Exception as e:
            print(f"Error adding to collection: {e}")

# Initialize the vector store manager
vectorstore = VectorStoreManager()

In [21]:
chunked_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data\\1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Br

In [22]:
# Generate embeddings for the chunked documents
chunk_embeddings = embedding_manager.embd_documents(chunked_documents)

# Add the documents and their embeddings to the vector store
vectorstore.add_documents(documents=chunked_documents, embeddings=chunk_embeddings)

Batches: 100%|██████████| 26/26 [00:04<00:00,  5.86it/s]


Generated embedding of shape (828, 384)
Adding 828 documents to vector space...
Successfully added the required fields in client


In [23]:
from langchain_core.documents import Document


class RAG_Retrival:
    def __init__(self, vector_collection: chromadb.Collection, embeddings: EmbeddingManager):
        self.collection = vector_collection
        self.embeddings = embeddings

    def retrieve(self, query: str, top_k: int = 5):
        """
        Retrieve relevant documents using a hybrid approach:
        1. Semantic search for top_k results.
        2. Keyword search for any word match.
        3. Combine and de-duplicate the results.
        """
        print(f"\n--- Retrieving for query: '{query}' ---")
        
        # 1. Semantic Search
        print("Performing semantic search...")
        query_embedding = self.embeddings.embd_documents([Document(page_content=query, metadata={})])
        semantic_results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )

        # 2. Keyword Search for ANY word match
        print("Performing keyword search...")
        keywords = [word for word in query.lower().split() if len(word) > 2] # Simple keyword extraction
        keyword_results = {'ids': [], 'documents': [], 'metadatas': []}
        if keywords:
            try:
                where_filter = {"$or": [{ "documents": {"$contains": keyword}} for keyword in keywords]}
                keyword_results = self.collection.get(where=where_filter, include=["metadatas", "documents"])
            except Exception as e:
                print(f"  [!] Error during keyword query: {e}")

        # 3. Combine and De-duplicate results
        print("Combining and de-duplicating results...")
        final_docs = {}

        # Add semantic results first
        if semantic_results and semantic_results['ids'][0]:
            for i, doc_id in enumerate(semantic_results['ids'][0]):
                if doc_id not in final_docs:
                    distance = semantic_results['distances'][0][i]
                    final_docs[doc_id] = {"content": semantic_results['documents'][0][i], "metadata": semantic_results['metadatas'][0][i], "source": "semantic"}

        # Add keyword results
        if keyword_results and keyword_results['ids']:
            for i, doc_id in enumerate(keyword_results['ids']):
                if doc_id not in final_docs:
                    final_docs[doc_id] = {"content": keyword_results['documents'][i], "metadata": keyword_results['metadatas'][i], "source": "keyword"}
        
        retrieved_docs = list(final_docs.values())
        print(f"Retrieved {len(retrieved_docs)} unique documents.")
        return retrieved_docs

# Pass the ChromaDB collection object directly
rag_retriever = RAG_Retrival(vectorstore.collection, embedding_manager)

In [24]:
rag_retriever.retrieve("Attention Is All You Need")


--- Retrieving for query: 'Attention Is All You Need' ---
Performing semantic search...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.77it/s]

Generated embedding of shape (1, 384)


Performing keyword search...
Combining and de-duplicating results...
Retrieved 5 unique documents.


[{'content': 'recurrent layers, by a factor of k. Separable convolutions [ 6], however, decrease the complexity\nconsiderably, to O(k · n · d + n · d2). Even with k = n, however, the complexity of a separable\nconvolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side benefit, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-',


Loading the model and generating thing like the convention input output

In [4]:
import torch
import time
# from google.colab import userdata
from transformers import AutoModelForCausalLM, AutoTokenizer

In [14]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"


tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto" # This automatically handles pushing it to GPU
)

Loading weights: 100%|██████████| 201/201 [00:05<00:00, 39.00it/s]


In [ ]:
prompt=""
print("Starting generation...")
start_time = time.perf_counter()
while prompt!="exit":
    prompt=input()
    RAG_context=rag_retriever.retrieve(prompt)
    print(f"Retrieved {len(RAG_context)} documents for the prompt.")
    rag_i=[rag["content"] for rag in RAG_context]
    prompt = ""
    print("Starting generation...")
    start_time = time.perf_counter()

    if rag_i:
            context_prompt =  prompt+ "\n\n".join(rag_i) + "\n\n" 
    else:
            print("No retrieved context available; generating from prompt only.")
            context_prompt = prompt

    inputs = tokenizer(
            context_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to("cuda")

    with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7,
                eos_token_id=tokenizer.eos_token_id
            )
    end_time = time.perf_counter()

    generation_time = end_time - start_time
    tokens_generated = len(outputs[0]) - len(inputs.input_ids[0])
    tokens_per_sec = tokens_generated / generation_time

    peak_vram = torch.cuda.max_memory_allocated("cuda") / (1024 ** 3)

    print("\n--- BENCHMARK RESULTS ---")
    print(f"Tokens Generated: {tokens_generated}")
    print(f"Execution Time: {generation_time:.2f} seconds")
    print(f"Throughput: {tokens_per_sec:.2f} tokens/sec")
    print(f"Peak GPU VRAM Usage: {peak_vram:.2f} GB")

    print("\n--- SAMPLE OUTPUT ---")
    print(tokenizer.decode(outputs[:512], skip_special_tokens=True))
    

Starting generation...

--- Retrieving for query: 'Teach me about the attention is all you need paper' ---
Performing semantic search...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.43s/it]


Generated embedding of shape (1, 384)
Performing keyword search...
Combining and de-duplicating results...
Retrieved 5 unique documents.
Retrieved 5 documents for the prompt.
Starting generation...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- BENCHMARK RESULTS ---
Tokens Generated: 175
Execution Time: 9.52 seconds
Throughput: 18.39 tokens/sec
Peak GPU VRAM Usage: 3.16 GB

--- SAMPLE OUTPUT ---
['Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nFigure 3: An example of the attention mechanism following long-distance dependencies in the\nencoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of\nthe verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for\nthe word ‘making’. Different colors represe

...

Generated embedding of shape (1, 384)
Performing keyword search...
Combining and de-duplicating results...
Retrieved 5 unique documents.
Retrieved 5 documents for the prompt.
Starting generation...

--- BENCHMARK RESULTS ---
Tokens Generated: 512
Execution Time: 20.13 seconds
Throughput: 25.43 tokens/sec
Peak GPU VRAM Usage: 3.16 GB

--- SAMPLE OUTPUT ---
['this waitlist were OpenAI employees, biasing the ultimate group toward our own networks.\nStepping back, there are many difﬁculties in designing an alignment process that is fair, transparent,\nand has suitable accountability mechanisms in place. The goal of this paper is to demonstrate that\nthis alignment technique can align to an speciﬁc human reference group for a speciﬁc application.\nWe are not claiming that researchers, the labelers we hired, or our API customers are the right source\nof preferences. There are many stakeholders to consider—the organization training the model, the\ncustomers using the model to develop products

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated embedding of shape (1, 384)
Performing keyword search...
  [!] Error during keyword query: Expected where value for $and or $or to be a list with at least two where expressions, got [{'documents': {'$contains': 'exit'}}] in get.
Combining and de-duplicating results...
Retrieved 5 unique documents.
Retrieved 5 documents for the prompt.
Starting generation...

--- BENCHMARK RESULTS ---
Tokens Generated: 512
Execution Time: 20.90 seconds
Throughput: 24.50 tokens/sec
Peak GPU VRAM Usage: 3.16 GB

--- SAMPLE OUTPUT ---
['74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n74\n\n

Batches: 100%|██████████| 1/1 [00:00<00:00, 95.54it/s]
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated embedding of shape (1, 384)
Performing keyword search...
Combining and de-duplicating results...
Retrieved 5 unique documents.
Retrieved 5 documents for the prompt.
Starting generation...

--- BENCHMARK RESULTS ---
Tokens Generated: 512
Execution Time: 22.29 seconds
Throughput: 22.97 tokens/sec
Peak GPU VRAM Usage: 3.16 GB

--- SAMPLE OUTPUT ---
['this waitlist were OpenAI employees, biasing the ultimate group toward our own networks.\nStepping back, there are many difﬁculties in designing an alignment process that is fair, transparent,\nand has suitable accountability mechanisms in place. The goal of this paper is to demonstrate that\nthis alignment technique can align to an speciﬁc human reference group for a speciﬁc application.\nWe are not claiming that researchers, the labelers we hired, or our API customers are the right source\nof preferences. There are many stakeholders to consider—the organization training the model, the\ncustomers using the model to develop products